# 

In [1]:
# 1. Импорты и общие настройки
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import transforms, datasets, models
from torchvision.models import ResNet18_Weights
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.datasets import VOCDetection
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple
import time

# Настройки
RANDOM_STATE = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

# Пути для артефактов
ARTIFACTS_DIR = "artifacts"
FIGURES_DIR = os.path.join(ARTIFACTS_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

Device: cuda
PyTorch: 2.7.1+cu118


In [2]:
# 2. Конфигурация
@dataclass
class ExperimentConfig:
    # Общие
    seed: int = 42
    batch_size: int = 64
    num_workers: int = 2
    
    # Классификация - ЧАСТЬ A
    clf_epochs: int = 5
    clf_lr: float = 1e-3
    input_size: int = 32
    dataset_a: str = "CIFAR100" 
    num_classes_a: int = 100
    
    # Detection - ЧАСТЬ B
    seg_batch_size: int = 4
    dataset_b: str = "PASCALVOC" 
    track_b: str = "detection"   
    score_threshold_v1: float = 0.3
    score_threshold_v2: float = 0.7
    foreground_class: int = 15    # Person class in VOC
    
    # Режимы
    fast_mode: bool = True

cfg = ExperimentConfig()
print(asdict(cfg))

{'seed': 42, 'batch_size': 64, 'num_workers': 2, 'clf_epochs': 5, 'clf_lr': 0.001, 'input_size': 32, 'dataset_a': 'CIFAR100', 'num_classes_a': 100, 'seg_batch_size': 4, 'dataset_b': 'PASCALVOC', 'track_b': 'detection', 'score_threshold_v1': 0.3, 'score_threshold_v2': 0.7, 'foreground_class': 15, 'fast_mode': True}


In [3]:
# 3. Часть A: Данные (CIFAR10)
# Нормализация для CIFAR-100
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD = (0.2675, 0.2565, 0.2761)

# Нормализация для ImageNet 
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# Трансформы для CNN с нуля
transform_base = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

# Трансформ для ResNet18 (ImageNet stats)
transform_imagenet = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Загрузка датасета
ds_train_full = datasets.CIFAR100(root="./data", train=True, download=True, transform=transform_base)
ds_test = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform_base)

# Для аугментаций
ds_train_aug = datasets.CIFAR100(root="./data", train=True, download=True, transform=transform_aug)
ds_train_imagenet = datasets.CIFAR100(root="./data", train=True, download=True, transform=transform_imagenet)
ds_test_imagenet = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform_imagenet)

# Разделение Train/Val
n_total = len(ds_train_full)
n_val = int(n_total * 0.2)
n_train = n_total - n_val

generator = torch.Generator().manual_seed(RANDOM_STATE)
train_subset, val_subset = random_split(ds_train_full, [n_train, n_val], generator=generator)
train_aug_subset, val_aug_subset = random_split(ds_train_aug, [n_train, n_val], generator=generator)

# DataLoaders
train_loader = DataLoader(train_subset, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
val_loader = DataLoader(val_subset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
test_loader = DataLoader(ds_test, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

print(f"Train: {len(train_subset)}, Val: {len(val_subset)}, Test: {len(ds_test)}")

D:\aie\dpo-dzhoshkun-group2\homeworks\HW10-11\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train: 40000, Val: 10000, Test: 10000


In [4]:
fig, axes = plt.subplots(2, 5, figsize=(20, 4))

# Original images
for i, ax in enumerate(axes[0]):
    if i < len(ds_train_full):
        img, label = ds_train_full[i]
        img_np = img.permute(1, 2, 0).numpy()
        img_np = img_np * CIFAR100_STD + CIFAR100_MEAN
        img_np = np.clip(img_np, 0, 1)
        ax.imshow(img_np)
        ax.set_title(f"Original {label}")
        ax.axis('off')

# Augmented images
for i, ax in enumerate(axes[1]):
    if i < len(ds_train_aug):
        img, label = ds_train_aug[i]
        img_np = img.permute(1, 2, 0).numpy()
        img_np = img_np * CIFAR100_STD + CIFAR100_MEAN
        img_np = np.clip(img_np, 0, 1)
        ax.imshow(img_np)
        ax.set_title(f"Aug {label}")
        ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "augmentations_preview.png"), dpi=150)
plt.close()
print("Saved augmentations_preview.png")


Saved augmentations_preview.png


In [5]:
# Simple CNN
class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 100):  # CIFAR100
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# ResNet18 с правильными weights
def build_resnet18(num_classes: int = 100):
    weights = ResNet18_Weights.DEFAULT 
    model = models.resnet18(weights=weights)
    
    # Замена головы
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model, weights

In [6]:
# 6. Часть A: Утилиты обучения
def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_correct, total_seen = 0.0, 0, 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        
        if not torch.isfinite(loss):
            return float("nan"), float("nan")
        
        loss.backward()
        optimizer.step()
        
        bs = y.size(0)
        total_loss += loss.item() * bs
        total_correct += (torch.argmax(logits, dim=1) == y).sum().item()
        total_seen += bs
    
    return total_loss / total_seen, total_correct / total_seen

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_seen = 0.0, 0, 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        
        bs = y.size(0)
        total_loss += loss.item() * bs
        total_correct += (torch.argmax(logits, dim=1) == y).sum().item()
        total_seen += bs
    
    return total_loss / total_seen, total_correct / total_seen

def fit(model, train_loader, val_loader, optimizer, criterion, epochs, device, exp_id):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0
    best_model_state = None
    
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)
        
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        
        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_model_state = model.state_dict().copy()
        
        dt = time.time() - t0
        print(f"Epoch {epoch:02d}/{epochs}| {exp_id}| train loss {tr_loss:.4f}, acc {tr_acc:.3f}| val loss {va_loss:.4f}, acc {va_acc:.3f}| {dt:.1f}s")
        
        if (not np.isfinite(tr_loss)) or (not np.isfinite(va_loss)):
            print("NaN/Inf в loss – останавливаем обучение.")
            break
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return history, best_val_acc

In [7]:
# 7. Часть A: Эксперименты C1-C4
results_a = []
criterion = nn.CrossEntropyLoss()

# C1: Simple CNN с нуля
print("\n=== Эксперимент C1: Simple CNN с нуля ===")
model_c1 = SimpleCNN(num_classes=cfg.num_classes_a).to(DEVICE)
optimizer_c1 = torch.optim.Adam(model_c1.parameters(), lr=cfg.clf_lr)
hist_c1, best_acc_c1 = fit(model_c1, train_loader, val_loader, optimizer_c1, criterion, cfg.clf_epochs, DEVICE, "C1")
test_acc_c1 = evaluate(model_c1, test_loader, criterion, DEVICE)[1]
results_a.append({"experiment_id": "C1", "task": "classification", "dataset": cfg.dataset_a, 
                  "best_val_accuracy": round(best_acc_c1, 4), "test_accuracy": round(test_acc_c1, 4)})

# C2: Simple CNN + Augmentations
print("\n=== Эксперимент C2: Simple CNN + Augmentations ===")
train_loader_aug = DataLoader(train_aug_subset, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
val_loader_aug = DataLoader(val_aug_subset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
model_c2 = SimpleCNN(num_classes=cfg.num_classes_a).to(DEVICE)
optimizer_c2 = torch.optim.Adam(model_c2.parameters(), lr=cfg.clf_lr)
hist_c2, best_acc_c2 = fit(model_c2, train_loader_aug, val_loader_aug, optimizer_c2, criterion, cfg.clf_epochs, DEVICE, "C2")
test_acc_c2 = evaluate(model_c2, test_loader, criterion, DEVICE)[1]
results_a.append({"experiment_id": "C2", "task": "classification", "dataset": cfg.dataset_a, 
                  "best_val_accuracy": round(best_acc_c2, 4), "test_accuracy": round(test_acc_c2, 4)})

# C3: ResNet18 - Freeze Backbone (head-only)
print("\n=== Эксперимент C3: ResNet18 - Freeze Backbone (head-only) ===")
model_c3, weights = build_resnet18(num_classes=cfg.num_classes_a)
model_c3 = model_c3.to(DEVICE)

# Заморозка backbone: явная заморозка
for param in model_c3.parameters():
    param.requires_grad = False
for param in model_c3.fc.parameters():
    param.requires_grad = True

print(f"Trainable params (C3 head-only): {count_params(model_c3)}")

# Используем правильный preprocessing для pretrained
train_loader_tl = DataLoader(random_split(ds_train_imagenet, [n_train, n_val], generator=generator)[0], 
                              batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
val_loader_tl = DataLoader(random_split(ds_train_imagenet, [n_train, n_val], generator=generator)[1], 
                            batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

optimizer_c3 = torch.optim.Adam(model_c3.fc.parameters(), lr=cfg.clf_lr)
hist_c3, best_acc_c3 = fit(model_c3, train_loader_tl, val_loader_tl, optimizer_c3, criterion, cfg.clf_epochs, DEVICE, "C3")
test_acc_c3 = evaluate(model_c3, DataLoader(ds_test_imagenet, batch_size=cfg.batch_size, shuffle=False), criterion, DEVICE)[1]
results_a.append({"experiment_id": "C3", "task": "classification", "dataset": cfg.dataset_a, 
                  "best_val_accuracy": round(best_acc_c3, 4), "test_accuracy": round(test_acc_c3, 4)})

# C4: ResNet18 - Fine-tuning Layer4
print("\n=== Эксперимент C4: ResNet18 - Fine-tuning Layer4 ===")
model_c4, weights = build_resnet18(num_classes=cfg.num_classes_a)
model_c4 = model_c4.to(DEVICE)

# Fine-tuning: размораживаем layer4
for param in model_c4.parameters():
    param.requires_grad = False
for param in model_c4.layer4.parameters():
    param.requires_grad = True
for param in model_c4.fc.parameters():
    param.requires_grad = True

print(f"Trainable params (C4 layer4+fc): {count_params(model_c4)}")

optimizer_c4 = torch.optim.Adam([
    {"params": model_c4.layer4.parameters(), "lr": 1e-4},
    {"params": model_c4.fc.parameters(), "lr": cfg.clf_lr}
])
hist_c4, best_acc_c4 = fit(model_c4, train_loader_tl, val_loader_tl, optimizer_c4, criterion, cfg.clf_epochs, DEVICE, "C4")
test_acc_c4 = evaluate(model_c4, DataLoader(ds_test_imagenet, batch_size=cfg.batch_size, shuffle=False), criterion, DEVICE)[1]
results_a.append({"experiment_id": "C4", "task": "classification", "dataset": cfg.dataset_a, 
                  "best_val_accuracy": round(best_acc_c4, 4), "test_accuracy": round(test_acc_c4, 4)})

# Сохраняем лучшую модель
torch.save(model_c4.state_dict(), os.path.join(ARTIFACTS_DIR, "best_classifier.pt"))
print(f"Saved best_classifier.pt")

# Сохраняем конфиг
config_dict = {
    "experiment_id": "C4",
    "model": "ResNet18",
    "dataset": cfg.dataset_a, 
    "seed": cfg.seed,
    "epochs": cfg.clf_epochs,
    "batch_size": cfg.batch_size,
    "transforms": {
        "base": ["ToTensor", "Normalize"],
        "augmentation": ["RandomHorizontalFlip", "RandomCrop"]
    },
    "preprocessing": "ResNet18_Weights.DEFAULT"
}
with open(os.path.join(ARTIFACTS_DIR, "best_classifier_config.json"), "w") as f:
    json.dump(config_dict, f, indent=4)
print(f"Saved best_classifier_config.json")



=== Эксперимент C1: Simple CNN с нуля ===
Epoch 01/5| C1| train loss 3.6695, acc 0.143| val loss 3.1419, acc 0.230| 16.5s
Epoch 02/5| C1| train loss 2.8626, acc 0.283| val loss 2.7631, acc 0.308| 15.8s
Epoch 03/5| C1| train loss 2.4749, acc 0.362| val loss 2.5806, acc 0.348| 15.9s
Epoch 04/5| C1| train loss 2.1945, acc 0.423| val loss 2.4220, acc 0.385| 15.6s
Epoch 05/5| C1| train loss 1.9724, acc 0.474| val loss 2.3900, acc 0.399| 15.8s

=== Эксперимент C2: Simple CNN + Augmentations ===
Epoch 01/5| C2| train loss 3.8883, acc 0.100| val loss 3.4768, acc 0.165| 17.1s
Epoch 02/5| C2| train loss 3.2750, acc 0.201| val loss 3.1949, acc 0.220| 17.2s
Epoch 03/5| C2| train loss 2.9868, acc 0.255| val loss 2.9527, acc 0.266| 17.2s
Epoch 04/5| C2| train loss 2.7790, acc 0.294| val loss 2.7509, acc 0.306| 17.1s
Epoch 05/5| C2| train loss 2.6152, acc 0.330| val loss 2.6519, acc 0.324| 17.1s

=== Эксперимент C3: ResNet18 - Freeze Backbone (head-only) ===
Trainable params (C3 head-only): 51300
Ep

In [8]:
# 8. Часть B: Detection (PASCAL VOC)
print("\n=== Часть B: Detection ===")

# Загрузка VOC Detection
try:
    ds_voc_train = VOCDetection(root="./data", year="2007", image_set="train", download=True)
    ds_voc_val = VOCDetection(root="./data", year="2007", image_set="val", download=True)
    print(f"VOC Detection loaded: {len(ds_voc_train)} train, {len(ds_voc_val)} val")
except Exception as e:
    print(f"Warning: VOCDetection download failed: {e}")
    ds_voc_train = None
    ds_voc_val = None

# Модель Detection
weights_det = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model_det = fasterrcnn_resnet50_fpn(weights=weights_det)
model_det = model_det.to(DEVICE)
model_det.eval()

# Функция для расчета IoU
def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

# Эксперименты V1 и V2 с разными threshold
def evaluate_detection(model, dataset, score_threshold, device, n_samples=10):
    model.eval()
    total_iou = 0
    total_tp = 0
    total_fp = 0
    total_fn = 0
    count = 0
    
    vis_indices = []
    vis_images = []
    vis_preds = []
    vis_gts = []
    
    if dataset is None:
        return 0, 0, 0, [], [], [], []
    
    for idx in range(min(n_samples, len(dataset))):
        try:
            item = dataset[idx]
            img_pil = item[0]
            target = item[1]
            
            img_tensor = transforms.ToTensor()(img_pil).to(device)
            
            with torch.no_grad():
                preds = model([img_tensor])[0]
            
            # Фильтрация по threshold (V1=0.3, V2=0.7)
            keep = preds["scores"] >= score_threshold
            pred_boxes = preds["boxes"][keep].cpu().numpy()
            pred_scores = preds["scores"][keep].cpu().numpy()
            
            gt_boxes = target["boxes"].numpy() if len(target["boxes"]) > 0 else np.array([])
            
            img_iou = 0
            if len(pred_boxes) > 0 and len(gt_boxes) > 0:
                gt = gt_boxes[0]
                best_iou = 0
                for pb in pred_boxes:
                    iou = calculate_iou(pb, gt)
                    if iou > best_iou:
                        best_iou = iou
                img_iou = best_iou
                if best_iou >= 0.5:
                    total_tp += 1
                else:
                    total_fp += 1
            elif len(gt_boxes) > 0:
                total_fn += 1
            
            total_iou += img_iou
            count += 1
            
            vis_indices.append(idx)
            vis_images.append(np.array(img_pil))
            vis_preds.append(pred_boxes)
            vis_gts.append(gt_boxes)
        except Exception as e:
            print(f"Error processing sample {idx}: {e}")
            continue
    
    mean_iou = total_iou / count if count > 0 else 0
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    
    return precision, recall, mean_iou, vis_indices, vis_images, vis_preds, vis_gts

# V1: threshold = 0.3
print("\n=== Эксперимент V1: Detection threshold=0.3 ===")
p1, r1, iou1, idx1, img1, pred1, gt1 = evaluate_detection(model_det, ds_voc_train, cfg.score_threshold_v1, DEVICE, n_samples=5)
print(f"V1 (0.3): Precision={p1:.2f}, Recall={r1:.2f}, IoU={iou1:.2f}")

# V2: threshold = 0.7
print("\n=== Эксперимент V2: Detection threshold=0.7 ===")
p2, r2, iou2, idx2, img2, pred2, gt2 = evaluate_detection(model_det, ds_voc_train, cfg.score_threshold_v2, DEVICE, n_samples=5)
print(f"V2 (0.7): Precision={p2:.2f}, Recall={r2:.2f}, IoU={iou2:.2f}")

results_b = [
    {"experiment_id": "V1", "task": cfg.track_b, "dataset": cfg.dataset_b, 
     "mean_iou": round(iou1, 4), "precision": round(p1, 4), "recall": round(r1, 4)},
    {"experiment_id": "V2", "task": cfg.track_b, "dataset": cfg.dataset_b, 
     "mean_iou": round(iou2, 4), "precision": round(p2, 4), "recall": round(r2, 4)}
]



=== Часть B: Detection ===


100.0%


VOC Detection loaded: 2501 train, 2510 val

=== Эксперимент V1: Detection threshold=0.3 ===
Error processing sample 0: 'boxes'
Error processing sample 1: 'boxes'
Error processing sample 2: 'boxes'
Error processing sample 3: 'boxes'
Error processing sample 4: 'boxes'
V1 (0.3): Precision=0.00, Recall=0.00, IoU=0.00

=== Эксперимент V2: Detection threshold=0.7 ===
Error processing sample 0: 'boxes'
Error processing sample 1: 'boxes'
Error processing sample 2: 'boxes'
Error processing sample 3: 'boxes'
Error processing sample 4: 'boxes'
V2 (0.7): Precision=0.00, Recall=0.00, IoU=0.00


In [9]:
# 9. Сохранение артефактов Detection
    n = min(len(images), 3)
    if n == 0:
        return
    
    fig, axes = plt.subplots(n, 3, figsize=(15, 5*n))
    if n == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(n):
        img = images[i]
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"Image {indices[i]}")
        axes[i, 0].axis('off')
        
        # GT
        axes[i, 1].imshow(img)
        if len(gts[i]) > 0:
            for box in gts[i][:3]:
                rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], 
                                     linewidth=2, edgecolor='g', facecolor='none')
                axes[i, 1].add_patch(rect)
        axes[i, 1].set_title("Ground Truth")
        axes[i, 1].axis('off')
        
        # Pred
        axes[i, 2].imshow(img)
        if len(preds[i]) > 0:
            for box in preds[i][:3]:
                rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], 
                                     linewidth=2, edgecolor='r', facecolor='none')
                axes[i, 2].add_patch(rect)
        axes[i, 2].set_title("Prediction")
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(figures_dir, filename), dpi=150)
    plt.close()
    print(f"Saved {filename}")

# Сохраняем detection примеры - ИСПРАВЛЕНО
plot_detection(img1, pred1, gt1, idx1, "detection_examples.png", FIGURES_DIR)

# Сохраняем detection метрики - ИСПРАВЛЕНО
plt.figure(figsize=(8, 5))
plt.bar(['V1 (0.3)', 'V2 (0.7)'], [iou1, iou2], color=['blue', 'orange'])
plt.ylabel('Mean IoU')
plt.title('Detection Metrics Comparison (V1 vs V2)')
plt.ylim(0, 1)
for i, v in enumerate([iou1, iou2]):
    plt.text(i, v + 0.02, f'{v:.2f}', ha='center')
plt.savefig(os.path.join(FIGURES_DIR, "detection_metrics.png"), dpi=150)
plt.close()
print("Saved detection_metrics.png")

Saved detection_metrics.png


In [10]:
# 10. Сохранение runs.csv
all_results = results_a + results_b
df_results = pd.DataFrame(all_results)
df_results.to_csv(os.path.join(ARTIFACTS_DIR, "runs.csv"), index=False)
print(f"\nruns.csv saved with {len(df_results)} experiments")
print(df_results.to_string())


runs.csv saved with 6 experiments
  experiment_id            task    dataset  best_val_accuracy  test_accuracy  mean_iou  precision  recall
0            C1  classification   CIFAR100             0.3993         0.3987       NaN        NaN     NaN
1            C2  classification   CIFAR100             0.3241         0.3535       NaN        NaN     NaN
2            C3  classification   CIFAR100             0.6439         0.5765       NaN        NaN     NaN
3            C4  classification   CIFAR100             0.9395         0.7146       NaN        NaN     NaN
4            V1       detection  PASCALVOC                NaN            NaN       0.0        0.0     0.0
5            V2       detection  PASCALVOC                NaN            NaN       0.0        0.0     0.0
